# 👀 Multilayer perceptron (MLP)

In this notebook, we'll walk through the steps required to train your own multilayer perceptron on the CIFAR dataset

**Assignment 1:** the textbook model is built in **3 versions** that differ only in the activation function of the two hidden layers:

1. **ReLU** (the textbook's original model)
2. **LeakyReLU**
3. **Sigmoid**

Everything else (data, architecture, optimizer, learning rate, batch size, epochs and starting weights) is kept the same as the textbook,
so any difference in output and accuracy comes from the activation function. The interpretation is in section 5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models, optimizers, utils, datasets

try:
    from notebooks.utils import display  # textbook helper (works inside the textbook repo)
except ImportError:
    # same helper as notebooks/utils.py, so the notebook also runs standalone (e.g. in Google Colab)
    def display(images, n=10, size=(20, 3), cmap="gray_r", as_type="float32", save_to=None):
        if images.max() > 1.0:
            images = images / 255.0
        elif images.min() < 0.0:
            images = (images + 1.0) / 2.0
        plt.figure(figsize=size)
        for i in range(n):
            _ = plt.subplot(1, n, i + 1)
            plt.imshow(images[i].astype(as_type), cmap=cmap)
            plt.axis("off")
        if save_to:
            plt.savefig(save_to)
        plt.show()

## 0. Parameters <a name="parameters"></a>

In [ ]:
NUM_CLASSES = 10
BATCH_SIZE = 32        # as in the textbook
EPOCHS = 10            # as in the textbook
LEARNING_RATE = 0.0005 # as in the textbook
SEED = 42              # same starting weights for all three models

## 1. Prepare the Data <a name="prepare"></a>

In [ ]:
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = utils.to_categorical(y_train, NUM_CLASSES)
y_test = utils.to_categorical(y_test, NUM_CLASSES)

In [ ]:
display(x_train[:10])
print(y_train[:10])

## 2. Build the model <a name="build"></a>
The textbook model, with the hidden-layer activation passed in as a parameter. The random seed is reset before each build,
so all three versions start from the same initial weights.

In [ ]:
def build_model(activation, name):
    tf.keras.utils.set_random_seed(SEED)
    input_layer = layers.Input((32, 32, 3))

    x = layers.Flatten()(input_layer)
    x = layers.Dense(200)(x)
    x = activation()(x)
    x = layers.Dense(150)(x)
    x = activation()(x)

    output_layer = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return models.Model(input_layer, output_layer, name=name)

### Version 1: ReLU

In [ ]:
model_relu = build_model(lambda: layers.ReLU(), "ReLU")
model_relu.summary()

### Version 2: LeakyReLU

In [ ]:
model_leaky = build_model(lambda: layers.LeakyReLU(negative_slope=0.01), "LeakyReLU")
model_leaky.summary()

### Version 3: Sigmoid

In [ ]:
model_sigmoid = build_model(lambda: layers.Activation("sigmoid"), "Sigmoid")
model_sigmoid.summary()

In [ ]:
mlps = {"ReLU": model_relu, "LeakyReLU": model_leaky, "Sigmoid": model_sigmoid}

## 3. Train the model <a name="train"></a>
Each version is compiled and trained exactly as in the textbook. The test set is passed as validation data only to record the test accuracy after every epoch.

In [ ]:
histories = {}
for name, model in mlps.items():
    print(f"\n========== {name} ==========")
    opt = optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(
        loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"]
    )
    histories[name] = model.fit(
        x_train, y_train, batch_size=BATCH_SIZE, epochs=EPOCHS, shuffle=True,
        validation_data=(x_test, y_test), verbose=2,
    ).history

## 4. Evaluation <a name="evaluate"></a>

In [ ]:
results = {}
for name, model in mlps.items():
    print(name)
    results[name] = model.evaluate(x_test, y_test)

In [ ]:
print(f"{'Activation':<11}{'Epoch-1 test acc':>18}{'Final train acc':>17}{'Test acc':>10}{'Test loss':>11}")
for name, (loss, acc) in results.items():
    h = histories[name]
    print(f"{name:<11}{h['val_accuracy'][0]:>18.4f}{h['accuracy'][-1]:>17.4f}{acc:>10.4f}{loss:>11.4f}")

In [ ]:
colors = {"ReLU": "tab:blue", "LeakyReLU": "tab:green", "Sigmoid": "tab:red"}
ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for name, h in histories.items():
    ax[0].plot(ep, h["loss"], color=colors[name], label=f"{name} train")
    ax[0].plot(ep, h["val_loss"], color=colors[name], ls="--", label=f"{name} test")
    ax[1].plot(ep, h["accuracy"], color=colors[name], label=f"{name} train")
    ax[1].plot(ep, h["val_accuracy"], color=colors[name], ls="--", label=f"{name} test")
ax[0].set(title="Loss", xlabel="Epoch", ylabel="Categorical cross-entropy"); ax[0].legend(fontsize=8)
ax[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Output of the models

In [ ]:
CLASSES = np.array(
    [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]
)

actual_single = CLASSES[np.argmax(y_test, axis=-1)]
preds, preds_single = {}, {}
for name, model in mlps.items():
    preds[name] = model.predict(x_test)
    preds_single[name] = CLASSES[np.argmax(preds[name], axis=-1)]

In [ ]:
for name in mlps:
    conf = preds[name].max(axis=1)
    ok = preds_single[name] == actual_single
    print(f"{name:<10} mean top-class probability: correct {conf[ok].mean():.3f} | wrong {conf[~ok].mean():.3f}"
          f" | predictions >0.9 confident: {(conf > 0.9).mean():.1%}")

print("\nPer-class test accuracy:")
print(f"{'class':<12}" + "".join(f"{n:>11}" for n in mlps))
for c in CLASSES:
    m = actual_single == c
    print(f"{c:<12}" + "".join(f"{(preds_single[n][m] == c).mean():>11.3f}" for n in mlps))
print(f"\nReLU and LeakyReLU agree on {(preds_single['ReLU'] == preds_single['LeakyReLU']).mean():.1%} of test images")
print(f"ReLU and Sigmoid agree on   {(preds_single['ReLU'] == preds_single['Sigmoid']).mean():.1%} of test images")

In [ ]:
np.random.seed(SEED)
n_to_show = 10
indices = np.random.choice(range(len(x_test)), n_to_show)

fig = plt.figure(figsize=(15, 4))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, idx in enumerate(indices):
    img = x_test[idx]
    ax = fig.add_subplot(1, n_to_show, i + 1)
    ax.axis("off")
    ax.text(0.5, -0.30, "act = " + str(actual_single[idx]), fontsize=10, ha="center", transform=ax.transAxes, weight="bold")
    for j, name in enumerate(mlps):
        p = preds_single[name][idx]
        ax.text(0.5, -0.55 - 0.22 * j, f"{name[:5]} = {p}", fontsize=9, ha="center", transform=ax.transAxes,
                color="green" if p == actual_single[idx] else "red")
    ax.imshow(img)
plt.show()

In [ ]:
idx = indices[0]
fig, ax = plt.subplots(1, 4, figsize=(16, 3))
ax[0].imshow(x_test[idx]); ax[0].axis("off"); ax[0].set_title(f"actual: {actual_single[idx]}")
for a, name in zip(ax[1:], mlps):
    a.bar(CLASSES, preds[name][idx], color=colors[name]); a.set_ylim(0, 1)
    a.set_title(f"{name}: {preds_single[name][idx]} ({preds[name][idx].max():.2f})")
    a.tick_params(axis="x", rotation=90)
plt.suptitle("Softmax output for one test image"); plt.tight_layout(); plt.show()

## 5. Interpretation of observations

### Accuracy

| Activation | Test acc after epoch 1 | Final train acc | Final test acc | Test loss |
|---|---|---|---|---|
| ReLU | 40.2% | 51.3% | 48.2% | 1.459 |
| LeakyReLU | 39.8% | 52.0% | **49.1%** | **1.435** |
| Sigmoid | 39.0% | 50.8% | 47.5% | 1.467 |

- **All three versions finish close together (1.6 points between best and worst).** LeakyReLU did best in this run, with the highest test accuracy and lowest test loss, followed by ReLU and then Sigmoid.
- **ReLU and LeakyReLU are effectively tied.** The 0.9-point gap between them is about the size of the epoch-to-epoch wobble in the test curves: LeakyReLU dropped from 48.1% to 47.6% between epochs 6 and 7, and ReLU from 47.7% to 47.5% between epochs 7 and 8. In an earlier run of the same code, ReLU finished ahead of LeakyReLU (48.6% vs 48.1%). The order flips from run to run, so neither is reliably better here.
- **Sigmoid is consistently last.** It has the lowest training accuracy and the highest training loss at every one of the 10 epochs, starts lowest after epoch 1 (39.0%), and finishes with the lowest test accuracy, in this run and the earlier one. The sigmoid's derivative is at most 0.25 and goes to zero when a unit saturates, so smaller gradients flow back to the first layer (the vanishing-gradient effect), and it learns more slowly. ReLU and LeakyReLU pass the gradient through unchanged (derivative = 1) for positive inputs.
- **The gap is small because of the Adam optimizer.** Adam rescales each weight's update by the size of its recent gradients, which makes up for most of Sigmoid's weaker gradients. With plain SGD, or a deeper network, Sigmoid falls much further behind. With only two hidden layers, vanishing gradients are not severe here.
- **LeakyReLU's small edge is plausible but not proven.** LeakyReLU keeps a small slope (0.01) for negative inputs, so units never stop learning ("dying ReLU"), and it also has the lowest final training loss (1.353). In a shallow network with a small learning rate (0.0005), dead units are rare, so the benefit is small, and here it is within run-to-run noise.

### Why all three are only around 48%
That is well above the 10% you'd get by guessing, but low for CIFAR-10. The limit comes from the MLP, not the activation. `Flatten` turns each 32×32×3 image into 3,072 separate numbers and throws away which pixels are next to each other. The model can't learn edges or shapes that appear anywhere in the image, so it relies mostly on overall color and background. Also, 615k of the 646k parameters sit in the first Dense layer, which makes the model easy to overfit. A convolutional network (the next notebook in the book) does much better on the same data.

### Overfitting
By epoch 10, training accuracy (50.8–52.0%) is ahead of test accuracy (47.5–49.1%), and training loss (1.35–1.38) is below test loss (1.44–1.47). ReLU's test loss barely improved after epoch 7 (1.478 → 1.482 → 1.465 → 1.459) while its training loss kept falling. The models are starting to memorize the training set, and more epochs would mainly widen this gap. Dropout, batch normalization or early stopping would help.

### Output (softmax probabilities)
- **The models are unsure of their answers.** When a prediction is right, the top class gets 0.57–0.59 probability on average. When it's wrong, it gets about 0.43. Only 3–5% of predictions are above 0.9. For the truck example, all three predicted "truck" (ReLU 0.93, LeakyReLU 0.77, Sigmoid 0.72), and all three put "automobile" second, a sensible mix-up.
- **Sigmoid is the least confident** (3.3% of predictions above 0.9, versus 4.4% for ReLU and 5.2% for LeakyReLU). It trained the least, so its outputs are softer.
- **Vehicles (and frogs) are easier than most animals.** Ship (61–69%), frog (62–63%) and automobile (56–65%) are recognized best. Bird (24–28%), cat (26–33%) and dog (35–38%) are the hardest. Animals in CIFAR-10 share similar colors and backgrounds, and in the sample images the mistakes followed the background: deer on grass were called frog or bird, and a dark cat was called a frog. The per-class pattern is nearly the same for all three activations, so it reflects the data and the MLP design, not the activation.
- **Similar accuracy, different predictions.** Even though their accuracies are within 1.6 points, ReLU agrees with LeakyReLU on only 68.1% of test images and with Sigmoid on 65.1%. In the 10 sample images, ReLU got 7 right, LeakyReLU 6 and Sigmoid 5, and not always the same ones (only Sigmoid recognized the first cat). Many images sit near the decision boundary, and a small change such as a different activation flips them. Averaging the three models (an ensemble) would likely beat any single one.

### Conclusion
With the textbook setup (shallow MLP, Adam optimizer), the activation choice made only a small difference to accuracy: **LeakyReLU ≈ ReLU > Sigmoid**. The two ReLU-type activations learned a little faster and reached higher accuracy than Sigmoid, but which of the two comes first changes between runs. Sigmoid was last in every run because of its smaller, saturating gradients, a problem that grows with deeper networks or plain SGD. That is why ReLU-type activations are the default for hidden layers, while sigmoid is now mostly used for probability outputs. The biggest limit here is not the activation but the MLP itself, which ignores the spatial structure of images.
